In [2]:
import pandas as pd

orders = pd.read_csv("orders.csv")
print(orders.head())


   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                  restaurant_name  
0               New Foods Chinese  
1  Ruchi Curry House Multicuisine  
2           Spice Kitchen Punjabi  
3          Darbar Kitchen Non-Veg  
4       Royal Eatery South Indian  


In [3]:
users = pd.read_json("users.json")
print(users.head())


   user_id    name       city membership
0        1  User_1    Chennai    Regular
1        2  User_2       Pune       Gold
2        3  User_3  Bangalore       Gold
3        4  User_4  Bangalore    Regular
4        5  User_5       Pune       Gold


In [5]:
import sqlite3

# Create / connect to database
conn = sqlite3.connect("food_delivery.db")
cursor = conn.cursor()

# Read SQL file
with open("restaurants.sql", "r") as f:
    sql_script = f.read()

# Execute SQL script (creates table + inserts data)
cursor.executescript(sql_script)

conn.commit()


In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())


[('restaurants',)]


In [7]:
restaurants = pd.read_sql_query(
    "SELECT * FROM restaurants",
    conn
)

print(restaurants.head())


   restaurant_id restaurant_name  cuisine  rating
0              1    Restaurant_1  Chinese     4.8
1              2    Restaurant_2   Indian     4.1
2              3    Restaurant_3  Mexican     4.3
3              4    Restaurant_4  Chinese     4.1
4              5    Restaurant_5  Chinese     4.8


In [8]:
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)


In [9]:
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)


In [10]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [12]:
print(final_df.columns)


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating'],
      dtype='object')


In [13]:
#Order Trends Over Time
final_df.groupby("order_date")["total_amount"].sum()


order_date
01-01-2023    25483.04
01-01-2024    17201.50
01-02-2023    21066.21
01-03-2023    32608.53
01-04-2023    26036.89
                ...   
31-05-2023    23824.71
31-07-2023    23816.09
31-08-2023    18901.45
31-10-2023    22383.13
31-12-2023    19813.33
Name: total_amount, Length: 366, dtype: float64

In [14]:
#User Behavior Patterns
final_df.groupby("user_id")["order_id"].count()


user_id
1        1
2       10
3        2
4        4
5        5
        ..
2996     3
2997     5
2998     6
2999     1
3000     2
Name: order_id, Length: 2883, dtype: int64

In [15]:
#City-wise & Cuisine-wise Performance
final_df.groupby("city")["total_amount"].sum()
final_df.groupby("cuisine")["total_amount"].sum()


cuisine
Chinese    1930504.65
Indian     1971412.58
Italian    2024203.80
Mexican    2085503.09
Name: total_amount, dtype: float64

In [17]:
#Membership Impact (Gold vs Regular)
final_df.groupby("membership")["total_amount"].mean()


membership
Gold       797.145556
Regular    805.158434
Name: total_amount, dtype: float64

In [20]:
final_df["order_date"] = pd.to_datetime(
    final_df["order_date"],
    format="%d-%m-%Y"
)


In [21]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Filter Gold members
gold_df = df[df["membership"] == "Gold"]

# City-wise total revenue from Gold members
city_revenue = (
    gold_df
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

# Print result
print(city_revenue)

# Get top city
top_city = city_revenue.idxmax()
top_revenue = city_revenue.max()

print(f"\nCity with highest Gold member revenue: {top_city}")
print(f"Total Revenue: {top_revenue}")


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

City with highest Gold member revenue: Chennai
Total Revenue: 1080909.79


In [22]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names (safe practice)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Calculate average order value by cuisine
avg_order_value = (
    df.groupby("cuisine")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

# Print all cuisines with average order value
print(avg_order_value)

# Get cuisine with highest average order value
top_cuisine = avg_order_value.idxmax()
top_value = avg_order_value.max()

print(f"\nCuisine with highest average order value: {top_cuisine}")
print(f"Average Order Value: {top_value:.2f}")


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

Cuisine with highest average order value: Mexican
Average Order Value: 808.02


In [23]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Calculate total order value per user
user_total = (
    df.groupby("user_id")["total_amount"]
    .sum()
)

# Filter users with total orders > 1000
high_value_users = user_total[user_total > 1000]

# Count distinct users
count_users = high_value_users.count()

print("Number of distinct users with total orders > ₹1000:", count_users)



Number of distinct users with total orders > ₹1000: 2544


In [24]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Create rating ranges
bins = [0, 3, 4, 5]
labels = ["0-3", "3-4", "4-5"]

df["rating_range"] = pd.cut(
    df["rating"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Calculate total revenue per rating range
rating_revenue = (
    df.groupby("rating_range")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

print(rating_revenue)


rating_range
4-5    4157357.01
3-4    3599248.98
0-3     255018.13
Name: total_amount, dtype: float64


In [25]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Filter Gold members
gold_df = df[df["membership"] == "Gold"]

# Calculate average order value by city
avg_order_value_city = (
    gold_df.groupby("city")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

print(avg_order_value_city)


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64


In [26]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Count distinct restaurants per cuisine
restaurants_count = df.groupby("cuisine")["restaurant_id"].nunique()

# Total revenue per cuisine
revenue = df.groupby("cuisine")["total_amount"].sum()

# Combine results
summary = pd.concat([restaurants_count, revenue], axis=1)
summary.columns = ["distinct_restaurants", "total_revenue"]

# Sort to find lowest restaurant count with high revenue
summary_sorted = summary.sort_values(
    ["distinct_restaurants", "total_revenue"],
    ascending=[True, False]
)

print(summary_sorted)


         distinct_restaurants  total_revenue
cuisine                                     
Chinese                   120     1930504.65
Italian                   126     2024203.80
Indian                    126     1971412.58
Mexican                   128     2085503.09


In [27]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Calculate percentage of Gold member orders
total_orders = len(df)
gold_orders = len(df[df["membership"] == "Gold"])

percentage = round((gold_orders / total_orders) * 100)

print(f"Percentage of orders by Gold members: {percentage}%")


Percentage of orders by Gold members: 50%


In [29]:
import pandas as pd

df = pd.read_csv("final_food_delivery_dataset.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

restaurants = [
    "Grand Cafe Punjabi",
    "Grand Restaurant South Indian",
    "Ruchi Mess Multicuisine",
    "Ruchi Foods Chinese"
]

subset = df[df["restaurant_name_x"].isin(restaurants)]

stats = (
    subset.groupby("restaurant_name_x")
    .agg(
        total_orders=("order_id", "count"),
        avg_order_value=("total_amount", "mean"),
        total_revenue=("total_amount", "sum")
    )
    .sort_values("avg_order_value", ascending=False)
)

print(stats)


                               total_orders  avg_order_value  total_revenue
restaurant_name_x                                                          
Ruchi Mess Multicuisine                  40       851.226250       34049.05
Grand Restaurant South Indian            29       842.567586       24434.46
Grand Cafe Punjabi                       32       765.409063       24493.09
Ruchi Foods Chinese                      19       686.603158       13045.46


In [30]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Define combinations
combinations = [
    ("Gold", "Indian"),
    ("Gold", "Italian"),
    ("Regular", "Indian"),
    ("Regular", "Chinese"),
]

# Calculate revenue for each combination
results = {}

for member, cuisine in combinations:
    revenue = df[
        (df["membership"] == member) & (df["cuisine"] == cuisine)
    ]["total_amount"].sum()
    results[f"{member} + {cuisine}"] = revenue

print(results)


{'Gold + Indian': 979312.31, 'Gold + Italian': 1005779.05, 'Regular + Indian': 992100.27, 'Regular + Chinese': 952790.9099999999}


In [31]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Convert order_date to datetime (DD-MM-YYYY)
df["order_date"] = pd.to_datetime(df["order_date"], format="%d-%m-%Y")

# Extract quarter
df["quarter"] = df["order_date"].dt.quarter

# Calculate total revenue per quarter
quarter_revenue = (
    df.groupby("quarter")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

print(quarter_revenue)


quarter
3    2037385.10
4    2018263.66
1    2010626.64
2    1945348.72
Name: total_amount, dtype: float64


In [32]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Count total orders by Gold members
gold_orders = len(df[df["membership"] == "Gold"])

print("Total orders by Gold members:", gold_orders)


Total orders by Gold members: 4987


In [33]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Calculate total revenue for Hyderabad
hyderabad_revenue = df[df["city"] == "Hyderabad"]["total_amount"].sum()

print(round(hyderabad_revenue))


1889367


In [34]:
import pandas as pd

# Load dataset
df = pd.read_csv("final_food_delivery_dataset.csv")

# Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Count orders by Gold members in Chennai
gold_chennai_orders = df[
    (df["membership"] == "Gold") & (df["city"] == "Chennai")
].shape[0]

print(gold_chennai_orders)


1337
